In [1]:
%pip install -q google-generativeai

Note: you may need to restart the kernel to use updated packages.


In [1]:
# ENVIRONMENT & CONFIGURATION

import os
import json
import time
from pathlib import Path

from google import genai
from google.genai import types

print("Python environment loaded successfully.")
print("MVP embedding generation notebook initialized.")

Python environment loaded successfully.
MVP embedding generation notebook initialized.


In [2]:
import os

print("GEMINI_API_KEY exists:", bool(os.getenv("GEMINI_API_KEY")))

GEMINI_API_KEY exists: True


In [3]:
# GEMINI API CONFIGURATION

API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    raise EnvironmentError(
        "GEMINI_API_KEY environment variable is not set."
    )

client = genai.Client(api_key=API_KEY)

print("Gemini API client initialized successfully.")

Gemini API client initialized successfully.


In [4]:
# EMBEDDING MODEL CONFIGURATION

MODEL_NAME = "gemini-embedding-001"

EMBEDDING_DIMENSION = 1536

# Maximum documents sent in one API request
BATCH_SIZE = 100

# Retry configuration
MAX_RETRIES = 5

# Wait between successful API batches.
# This is intentionally conservative for the free tier.
BATCH_WAIT_SECONDS = 70

print("=" * 60)
print("EMBEDDING CONFIGURATION")
print("=" * 60)

print(f"Model              : {MODEL_NAME}")
print(f"Embedding dimension : {EMBEDDING_DIMENSION}")
print(f"Batch size          : {BATCH_SIZE}")
print(f"Maximum retries     : {MAX_RETRIES}")
print(f"Batch wait          : {BATCH_WAIT_SECONDS} seconds")

EMBEDDING CONFIGURATION
Model              : gemini-embedding-001
Embedding dimension : 1536
Batch size          : 100
Maximum retries     : 5
Batch wait          : 70 seconds


In [5]:
# DATA & OUTPUT PATHS

BASE_DIR = Path(
    r"E:\Python Projects\SwifTBaSkeT-AI\data"
)

CORPUS_FILES = {
    "products": BASE_DIR / "products.jsonl",
    "orders": BASE_DIR / "orders.jsonl",
    "returns": BASE_DIR / "returns.jsonl"
}

# IMPORTANT:
# These are NEW MVP files.
# We are deliberately not overwriting the old
# 41,180-document embedding files.

MVP_OUTPUT_FILE = BASE_DIR / "mvp_embeddings.jsonl"

MVP_CHECKPOINT_FILE = BASE_DIR / "mvp_embedding_progress.json"

print("=" * 60)
print("PATH CONFIGURATION")
print("=" * 60)

print(f"Data directory      : {BASE_DIR}")
print(f"MVP embedding file  : {MVP_OUTPUT_FILE}")
print(f"MVP checkpoint file : {MVP_CHECKPOINT_FILE}")

PATH CONFIGURATION
Data directory      : E:\Python Projects\SwifTBaSkeT-AI\data
MVP embedding file  : E:\Python Projects\SwifTBaSkeT-AI\data\mvp_embeddings.jsonl
MVP checkpoint file : E:\Python Projects\SwifTBaSkeT-AI\data\mvp_embedding_progress.json


In [6]:
# SOURCE CORPUS VALIDATION

FULL_EXPECTED_COUNTS = {
    "products": 2314,
    "orders": 10000,
    "returns": 28866
}

print("=" * 60)
print("VALIDATING SOURCE CORPUS")
print("=" * 60)

for corpus_name, filepath in CORPUS_FILES.items():

    if not filepath.exists():
        raise FileNotFoundError(
            f"Missing corpus file: {filepath}"
        )

    count = 0

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                json.loads(line)
                count += 1

    expected = FULL_EXPECTED_COUNTS[corpus_name]

    print(
        f"{corpus_name.upper():10} : "
        f"{count:,} documents | "
        f"Expected: {expected:,}"
    )

    if count != expected:
        raise ValueError(
            f"{corpus_name}: expected {expected}, got {count}"
        )

print("\nSOURCE CORPUS VALIDATION: PASS")

VALIDATING SOURCE CORPUS
PRODUCTS   : 2,314 documents | Expected: 2,314
ORDERS     : 10,000 documents | Expected: 10,000
RETURNS    : 28,866 documents | Expected: 28,866

SOURCE CORPUS VALIDATION: PASS


In [7]:
# MVP CORPUS SELECTION

MVP_COUNTS = {
    "products": 2314,
    "orders": 5000,
    "returns": 5000
}

corpora = {}

print("=" * 60)
print("LOADING MVP CORPUS")
print("=" * 60)

for corpus_name, filepath in CORPUS_FILES.items():

    print(f"\nLoading {corpus_name.upper()}...")

    documents = []

    with open(filepath, "r", encoding="utf-8") as f:

        for line in f:

            if not line.strip():
                continue

            doc = json.loads(line)
            documents.append(doc)

    requested_count = MVP_COUNTS[corpus_name]

    if len(documents) < requested_count:
        raise ValueError(
            f"{corpus_name} contains only {len(documents)} "
            f"documents, but {requested_count} were requested."
        )

    # Deterministic MVP selection.
    selected_documents = documents[:requested_count]

    corpora[corpus_name] = selected_documents

    print(
        f"  Full corpus     : {len(documents):,}"
    )

    print(
        f"  MVP documents   : {len(selected_documents):,}"
    )

# Combine the three corpora
all_documents = (
    corpora["products"]
    + corpora["orders"]
    + corpora["returns"]
)

print("\n" + "=" * 60)
print("MVP CORPUS SELECTION COMPLETED")
print("=" * 60)

print(f"Products : {len(corpora['products']):,}")
print(f"Orders   : {len(corpora['orders']):,}")
print(f"Returns  : {len(corpora['returns']):,}")
print(f"TOTAL    : {len(all_documents):,}")

if len(all_documents) != 12314:
    raise ValueError(
        f"Expected 12,314 MVP documents, "
        f"got {len(all_documents)}"
    )

print("\nMVP COUNT VERIFICATION: PASS")

LOADING MVP CORPUS

Loading PRODUCTS...
  Full corpus     : 2,314
  MVP documents   : 2,314

Loading ORDERS...
  Full corpus     : 10,000
  MVP documents   : 5,000

Loading RETURNS...
  Full corpus     : 28,866
  MVP documents   : 5,000

MVP CORPUS SELECTION COMPLETED
Products : 2,314
Orders   : 5,000
Returns  : 5,000
TOTAL    : 12,314

MVP COUNT VERIFICATION: PASS


In [8]:
# TEST EMBEDDING DIMENSION

EMBEDDING_DIMENSION = 1536

test_document = all_documents[0]["text"]

result_1536 = client.models.embed_content(
    model=MODEL_NAME,
    contents=test_document,
    config=types.EmbedContentConfig(
        task_type="RETRIEVAL_DOCUMENT",
        output_dimensionality=EMBEDDING_DIMENSION
    )
)

embedding_1536 = result_1536.embeddings[0].values

print("=" * 60)
print("REDUCED-DIMENSION EMBEDDING TEST")
print("=" * 60)
print(f"Model:              {MODEL_NAME}")
print(f"Requested dimension:{EMBEDDING_DIMENSION}")
print(f"Actual dimension:   {len(embedding_1536)}")
print(f"First 5 values:     {embedding_1536[:5]}")

if len(embedding_1536) != EMBEDDING_DIMENSION:
    raise ValueError(
        f"Dimension mismatch: expected {EMBEDDING_DIMENSION}, "
        f"got {len(embedding_1536)}"
    )

print("\nDimension verification: PASS")

REDUCED-DIMENSION EMBEDDING TEST
Model:              gemini-embedding-001
Requested dimension:1536
Actual dimension:   1536
First 5 values:     [0.013683371, 0.0106884595, 0.020693801, -0.033979334, 0.011101753]

Dimension verification: PASS


In [8]:
# MVP DOCUMENT VALIDATION

print("=" * 60)
print("MVP DOCUMENT VALIDATION")
print("=" * 60)

seen_ids = set()

for doc in all_documents:

    if "id" not in doc:
        raise ValueError("Document missing ID.")

    if "text" not in doc:
        raise ValueError(
            f"Document {doc['id']} missing text."
        )

    if not str(doc["text"]).strip():
        raise ValueError(
            f"Document {doc['id']} has empty text."
        )

    if "metadata" not in doc:
        raise ValueError(
            f"Document {doc['id']} missing metadata."
        )

    doc_id = str(doc["id"])

    if doc_id in seen_ids:
        raise ValueError(
            f"Duplicate document ID found: {doc_id}"
        )

    seen_ids.add(doc_id)

print(f"Documents validated : {len(all_documents):,}")
print(f"Unique IDs          : {len(seen_ids):,}")

if len(seen_ids) != len(all_documents):
    raise ValueError("Duplicate IDs detected.")

print("\nMVP DOCUMENT VALIDATION: PASS")

MVP DOCUMENT VALIDATION
Documents validated : 12,314
Unique IDs          : 12,314

MVP DOCUMENT VALIDATION: PASS


In [9]:
# MVP CORPUS CHARACTERISTICS

print("=" * 60)
print("MVP CORPUS CHARACTERISTICS")
print("=" * 60)

for corpus_name, docs in corpora.items():

    text_lengths = [
        len(str(doc.get("text", "")))
        for doc in docs
    ]

    print(f"\n{corpus_name.upper()}")
    print("-" * 40)

    print(f"Documents       : {len(docs):,}")
    print(f"Min text chars  : {min(text_lengths):,}")
    print(f"Max text chars  : {max(text_lengths):,}")
    print(
        f"Average chars   : "
        f"{sum(text_lengths) / len(text_lengths):,.0f}"
    )
    print(
        f"Total characters: "
        f"{sum(text_lengths):,}"
    )

print("\n" + "=" * 60)
print(f"TOTAL MVP DOCUMENTS: {len(all_documents):,}")
print("=" * 60)

MVP CORPUS CHARACTERISTICS

PRODUCTS
----------------------------------------
Documents       : 2,314
Min text chars  : 412
Max text chars  : 503
Average chars   : 452
Total characters: 1,045,424

ORDERS
----------------------------------------
Documents       : 5,000
Min text chars  : 646
Max text chars  : 3,834
Average chars   : 1,048
Total characters: 5,239,859

RETURNS
----------------------------------------
Documents       : 5,000
Min text chars  : 665
Max text chars  : 769
Average chars   : 705
Total characters: 3,523,179

TOTAL MVP DOCUMENTS: 12,314


In [11]:
# REDUCED DIMENSION EMBEDDING TEST

test_document = all_documents[0]["text"]

result = client.models.embed_content(
    model=MODEL_NAME,
    contents=test_document,
    config=types.EmbedContentConfig(
        task_type="RETRIEVAL_DOCUMENT",
        output_dimensionality=EMBEDDING_DIMENSION
    )
)

test_embedding = result.embeddings[0].values

print("=" * 60)
print("REDUCED-DIMENSION EMBEDDING TEST")
print("=" * 60)

print(f"Model              : {MODEL_NAME}")
print(f"Requested dimension : {EMBEDDING_DIMENSION}")
print(f"Actual dimension    : {len(test_embedding)}")
print(f"First 5 values      : {test_embedding[:5]}")

if len(test_embedding) != EMBEDDING_DIMENSION:
    raise ValueError(
        f"Dimension mismatch: expected "
        f"{EMBEDDING_DIMENSION}, "
        f"got {len(test_embedding)}"
    )

print("\nDimension verification: PASS")

REDUCED-DIMENSION EMBEDDING TEST
Model              : gemini-embedding-001
Requested dimension : 1536
Actual dimension    : 1536
First 5 values      : [0.013683371, 0.0106884595, 0.020693801, -0.033979334, 0.011101753]

Dimension verification: PASS


In [12]:
# BATCH EMBEDDING TEST

test_batch = all_documents[:5]

test_texts = [
    doc["text"]
    for doc in test_batch
]

batch_result = client.models.embed_content(
    model=MODEL_NAME,
    contents=test_texts,
    config=types.EmbedContentConfig(
        task_type="RETRIEVAL_DOCUMENT",
        output_dimensionality=EMBEDDING_DIMENSION
    )
)

batch_embeddings = batch_result.embeddings

print("=" * 60)
print("BATCH EMBEDDING TEST")
print("=" * 60)

print(f"Batch size          : {len(test_batch)}")
print(f"Embedding dimension : {EMBEDDING_DIMENSION}")
print(f"Model               : {MODEL_NAME}")
print(f"Embeddings returned : {len(batch_embeddings)}")

if len(batch_embeddings) != len(test_batch):
    raise ValueError(
        "Number of embeddings does not match "
        "number of documents."
    )

for i, embedding in enumerate(batch_embeddings):

    values = embedding.values

    if len(values) != EMBEDDING_DIMENSION:
        raise ValueError(
            f"Invalid dimension for document "
            f"{test_batch[i]['id']}"
        )

    print(
        f"Document {i+1}: "
        f"{test_batch[i]['id']} | "
        f"Dimensions: {len(values)} | "
        f"First 3: {values[:3]}"
    )

print("\nBATCH EMBEDDING TEST: PASS")

BATCH EMBEDDING TEST
Batch size          : 5
Embedding dimension : 1536
Model               : gemini-embedding-001
Embeddings returned : 5
Document 1: P000001 | Dimensions: 1536 | First 3: [0.013683371, 0.0106884595, 0.020693801]
Document 2: P000002 | Dimensions: 1536 | First 3: [-0.015656246, 0.027883053, 0.0017040103]
Document 3: P000003 | Dimensions: 1536 | First 3: [-0.0047480473, 0.013991199, 0.009917569]
Document 4: P000004 | Dimensions: 1536 | First 3: [0.014157193, 0.0035264883, 0.006503874]
Document 5: P000005 | Dimensions: 1536 | First 3: [0.00038198227, -0.0038757995, -0.0050073876]

BATCH EMBEDDING TEST: PASS


In [10]:
# CHECKPOINT & RESUME MANAGEMENT

def load_checkpoint():

    if MVP_CHECKPOINT_FILE.exists():

        with open(
            MVP_CHECKPOINT_FILE,
            "r",
            encoding="utf-8"
        ) as f:

            checkpoint = json.load(f)

        # Make sure checkpoint belongs to this MVP run.
        if (
            checkpoint.get("total_documents")
            != len(all_documents)
        ):
            print(
                "Existing checkpoint belongs to "
                "a different corpus size."
            )

            print(
                "Starting a fresh MVP checkpoint."
            )

            return {
                "total_documents": len(all_documents),
                "batch_size": BATCH_SIZE,
                "completed_documents": 0,
                "completed_batches": 0,
                "embedding_dimension": EMBEDDING_DIMENSION,
                "model": MODEL_NAME,
                "status": "NOT_STARTED"
            }

        return checkpoint

    return {
        "total_documents": len(all_documents),
        "batch_size": BATCH_SIZE,
        "completed_documents": 0,
        "completed_batches": 0,
        "embedding_dimension": EMBEDDING_DIMENSION,
        "model": MODEL_NAME,
        "status": "NOT_STARTED"
    }


def save_checkpoint(checkpoint):

    with open(
        MVP_CHECKPOINT_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            checkpoint,
            f,
            indent=4
        )


def get_existing_embedding_ids():

    ids = set()

    if not MVP_OUTPUT_FILE.exists():
        return ids

    with open(
        MVP_OUTPUT_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if not line.strip():
                continue

            record = json.loads(line)
            ids.add(str(record["id"]))

    return ids


print("Checkpoint management ready.")

Checkpoint management ready.


In [11]:
# EMBEDDING ENGINE WITH RETRY LOGIC

def get_embeddings_with_retry(
    texts,
    max_retries=MAX_RETRIES
):

    for attempt in range(1, max_retries + 1):

        try:

            response = client.models.embed_content(
                model=MODEL_NAME,
                contents=texts,
                config=types.EmbedContentConfig(
                    task_type="RETRIEVAL_DOCUMENT",
                    output_dimensionality=EMBEDDING_DIMENSION
                )
            )

            embeddings = [
                item.values
                for item in response.embeddings
            ]

            if len(embeddings) != len(texts):
                raise ValueError(
                    f"API returned {len(embeddings)} embeddings "
                    f"for {len(texts)} documents."
                )

            for embedding in embeddings:

                if len(embedding) != EMBEDDING_DIMENSION:
                    raise ValueError(
                        "Invalid embedding dimension returned."
                    )

            return embeddings

        except Exception as e:

            error_text = str(e).lower()

            is_rate_limit = (
                "429" in error_text
                or "resource_exhausted" in error_text
                or "too many requests" in error_text
            )

            if not is_rate_limit:
                print("\n[FATAL API ERROR]")
                print(e)
                raise

            if attempt == max_retries:
                print(
                    "\nMaximum retry attempts exceeded."
                )
                raise

            wait_time = BATCH_WAIT_SECONDS

            print(
                f"Rate limit encountered. "
                f"Waiting {wait_time} seconds "
                f"(attempt {attempt}/{max_retries})..."
            )

            time.sleep(wait_time)


print("Embedding engine with retry logic ready.")

Embedding engine with retry logic ready.


In [12]:
# MVP PRODUCTION EMBEDDING RUN

print("=" * 60)
print("SWIFTBASKET — MVP EMBEDDING RUN")
print("=" * 60)

checkpoint = load_checkpoint()

existing_ids = get_existing_embedding_ids()

print(f"Total MVP documents : {len(all_documents):,}")
print(f"Embedding dimension  : {EMBEDDING_DIMENSION}")
print(f"Batch size           : {BATCH_SIZE}")
print(f"Model                : {MODEL_NAME}")

print(
    f"\nExisting embeddings : "
    f"{len(existing_ids):,}"
)

remaining_documents = [
    doc
    for doc in all_documents
    if str(doc["id"]) not in existing_ids
]

print(
    f"Remaining documents : "
    f"{len(remaining_documents):,}"
)

if not remaining_documents:

    print("\nAll MVP documents are already embedded.")
    checkpoint["status"] = "COMPLETED"
    save_checkpoint(checkpoint)

else:

    batches = [
        remaining_documents[i:i + BATCH_SIZE]
        for i in range(
            0,
            len(remaining_documents),
            BATCH_SIZE
        )
    ]

    print(
        f"Remaining batches   : "
        f"{len(batches):,}"
    )

    checkpoint["status"] = "RUNNING"
    save_checkpoint(checkpoint)

    successful_batches = 0

    for batch_number, batch in enumerate(
        batches,
        start=1
    ):

        print("\n" + "-" * 60)

        print(
            f"MVP Batch "
            f"{batch_number}/{len(batches)}"
        )

        print(
            f"Documents : {len(batch)}"
        )

        texts = [
            doc["text"]
            for doc in batch
        ]

        try:

            embeddings = get_embeddings_with_retry(
                texts
            )

            with open(
                MVP_OUTPUT_FILE,
                "a",
                encoding="utf-8"
            ) as f:

                for doc, embedding in zip(
                    batch,
                    embeddings
                ):

                    record = {
                        "id": str(doc["id"]),
                        "text": doc["text"],
                        "embedding": embedding,
                        "metadata": doc["metadata"]
                    }

                    f.write(
                        json.dumps(
                            record,
                            ensure_ascii=False
                        )
                        + "\n"
                    )

            existing_ids.update(
                str(doc["id"])
                for doc in batch
            )

            successful_batches += 1

            checkpoint["completed_documents"] = (
                len(existing_ids)
            )

            checkpoint["completed_batches"] = (
                checkpoint.get(
                    "completed_batches",
                    0
                ) + 1
            )

            checkpoint["status"] = "RUNNING"

            save_checkpoint(checkpoint)

            print(
                f"SUCCESS: "
                f"{len(batch)} embeddings written."
            )

            print(
                f"Total completed: "
                f"{len(existing_ids):,}/"
                f"{len(all_documents):,}"
            )

            # Wait before the next API request.
            if batch_number < len(batches):

                print(
                    f"Waiting "
                    f"{BATCH_WAIT_SECONDS} seconds..."
                )

                time.sleep(
                    BATCH_WAIT_SECONDS
                )

        except Exception as e:

            checkpoint["status"] = "FAILED"
            save_checkpoint(checkpoint)

            print("\n" + "=" * 60)
            print("MVP EMBEDDING RUN STOPPED")
            print("=" * 60)

            print(
                f"Completed embeddings : "
                f"{len(existing_ids):,}"
            )

            print(
                f"Remaining            : "
                f"{len(all_documents) - len(existing_ids):,}"
            )

            print(f"\nError: {e}")

            print(
                "\nCheckpoint saved."
            )

            print(
                "You can safely rerun this cell later."
            )

            break

    else:

        checkpoint["status"] = "COMPLETED"
        checkpoint["completed_documents"] = len(
            existing_ids
        )

        save_checkpoint(checkpoint)

        print("\n" + "=" * 60)
        print("MVP EMBEDDING RUN COMPLETED")
        print("=" * 60)

        print(
            f"Total embeddings : "
            f"{len(existing_ids):,}"
        )

SWIFTBASKET — MVP EMBEDDING RUN
Total MVP documents : 12,314
Embedding dimension  : 1536
Batch size           : 100
Model                : gemini-embedding-001

Existing embeddings : 2,300
Remaining documents : 10,014
Remaining batches   : 101

------------------------------------------------------------
MVP Batch 1/101
Documents : 100
Rate limit encountered. Waiting 70 seconds (attempt 1/5)...


KeyboardInterrupt: 